# Bases de données — Exercice 6 : achats, articles et clients (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue : la requête SQL à taper dans la **console H2**, et le résultat obtenu |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** schéma relationnel, clé primaire composée de plusieurs attributs, clé étrangère, `CREATE TABLE` avec `PRIMARY KEY` et `FOREIGN KEY`, contrainte d'intégrité référentielle.

> On considère une base de données qui décrit des informations sur des articles et des clients qui achètent ces articles. Le schéma relationnel de cette base est le suivant, où la clé primaire de chaque relation est soulignée et les attributs qui forment une clé étrangère sont suivis de *.
>
> - Achats (<u>aid*, cid*, dateachat</u>, quantité)
> - Articles (<u>aid</u>, rayon, prix)
> - Clients (<u>cid</u>, nom, datenaiss)

---
# MISE EN PLACE

En classe, les requêtes SQL sont tapées dans la **console H2** (le SGBD utilisé dans le cours). C'est aussi le cas ici : **chaque requête de ce notebook a été exécutée dans H2**, et le résultat affiché sous la requête est celui qu'a donné H2. Il n'y a **aucun code à exécuter** dans ce notebook : il suffit de recopier les requêtes dans la console H2.

**Pour ouvrir la console H2 :** lancer « H2 Console », se connecter avec la JDBC URL `jdbc:h2:~/nsi`, l'utilisateur `sa` et un mot de passe vide.

> ⚠️ **Liberté prise :** les résultats ont été obtenus avec H2 version 2.5.252 et recopiés tels quels, à trois détails près : la durée d'exécution affichée par H2 (« 12 ms ») a été supprimée, car elle change à chaque fois ; pour les résultats longs, seules les 12 premières lignes sont recopiées, suivies de « ... » et du nombre total de lignes ; dans les messages d'erreur, le nom technique de l'erreur Java (`org.h2.jdbc....Exception`) a été remplacé par « Erreur : » ; quand plusieurs instructions sont exécutées d'un coup, chacune est rappelée au-dessus de son résultat. Dans la console du navigateur, le résultat apparaît sous forme de tableau ; le contenu est le même.

---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Est-il possible pour un client d'acheter un même article plusieurs fois ?

#### ✅ RÉSOLUTION

**Oui, mais à des dates différentes.**

La clé primaire de la relation `Achats` est le triplet (`aid`, `cid`, `dateachat`). Deux achats ne peuvent pas avoir le même triplet. Un même client (`cid`) peut donc acheter un même article (`aid`) plusieurs fois, **à condition que la date soit différente** à chaque fois.

En revanche, il ne peut pas y avoir deux enregistrements pour le même client, le même article et la même date. S'il achète deux fois le même article le même jour, on l'enregistre en une seule ligne, avec l'attribut `quantité`.

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Lire un schéma :** les attributs soulignés forment **ensemble** la clé primaire. Ici, ce n'est ni `aid` seul, ni `cid` seul, mais le **groupe des trois**.
- Si la clé primaire était seulement (`aid`, `cid`), un client ne pourrait acheter un article **qu'une seule fois dans toute sa vie** : ce serait absurde pour un magasin.
- L'attribut `quantité` répond au cas « plusieurs exemplaires le même jour ».
- `aid*` et `cid*` sont aussi des **clés étrangères** : `aid` fait référence à un article de la relation `Articles`, `cid` à un client de la relation `Clients`. Un attribut peut être à la fois dans la clé primaire et une clé étrangère.
- La question 3 montre, dans la console H2, que le SGBD accepte bien deux achats du même article à deux dates différentes, et refuse le même achat deux fois le même jour.

---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. Quelles sont les instructions SQL permettant de créer les relations ci-dessus avec H2 ?

#### ✅ RÉSOLUTION

On crée d'abord les deux relations référencées, puis `Achats`, qui fait référence aux deux autres :

```sql
CREATE TABLE Articles (
    aid INT,
    rayon VARCHAR,
    prix REAL,
    PRIMARY KEY (aid) );

CREATE TABLE Clients (
    cid INT,
    nom VARCHAR,
    datenaiss DATE,
    PRIMARY KEY (cid) );

CREATE TABLE Achats (
    aid INT,
    cid INT,
    dateachat DATE,
    quantité INT,
    PRIMARY KEY (aid, cid, dateachat),
    FOREIGN KEY (aid) REFERENCES Articles(aid),
    FOREIGN KEY (cid) REFERENCES Clients(cid) );
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Articles ( aid INT, rayon VARCHAR, prix REAL, PRIMARY KEY (aid) );
(Update count: 0)

CREATE TABLE Clients ( cid INT, nom VARCHAR, datenaiss DATE, PRIMARY KEY (cid) );
(Update count: 0)

CREATE TABLE Achats ( aid INT, cid INT, dateachat DATE, quantité INT, PRIMARY KEY (aid, cid, dateachat), FOREIGN KEY (aid) REFERENCES Articles(aid), FOREIGN KEY (cid) REFERENCES Clients(cid) );
(Update count: 0)
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Choisir les types :** les identifiants sont des entiers (`INT`), les textes des `VARCHAR`, le prix un nombre à virgule (`REAL`), les dates des `DATE` (en SQL, une date s'écrit `'AAAA-MM-JJ'`).
- **Clé primaire :** `PRIMARY KEY (aid)` ; pour une clé de plusieurs attributs, on les liste **tous dans les mêmes parenthèses** : `PRIMARY KEY (aid, cid, dateachat)`. ⚠️ Écrire trois fois `PRIMARY KEY` serait une erreur : une relation n'a qu'**une** clé primaire.
- **Clés étrangères :** `FOREIGN KEY (aid) REFERENCES Articles(aid)` signifie « la valeur de `aid` dans `Achats` doit exister dans l'attribut `aid` de `Articles` ».
- ⚠️ **L'ordre de création compte :** `Achats` fait référence à `Articles` et `Clients`, qui doivent donc **déjà exister**. Créer `Achats` en premier provoque une erreur dans H2 (« Table ARTICLES non trouvée »).
- L'accent de `quantité` est accepté par H2 ; on peut aussi écrire `quantite` pour éviter tout souci.

---
# QUESTION 3

#### 📝 ÉNONCÉ

> 3. On dispose de trois fichiers achats.sql, articles.sql et clients.sql qui permettent d'effectuer des enregistrements dans chacune des relations ci-dessus.
> On utilise la commande RUNSCRIPT pour exécuter les instructions SQL contenues dans ces fichiers : quel ordre chronologique peut-on choisir pour alimenter la base de données avec les fichiers mis à disposition ?

#### ✅ RÉSOLUTION

Il faut exécuter `articles.sql` et `clients.sql` **avant** `achats.sql` (l'ordre entre `articles.sql` et `clients.sql` n'a pas d'importance) :

```sql
RUNSCRIPT FROM 'articles.sql';
RUNSCRIPT FROM 'clients.sql';
RUNSCRIPT FROM 'achats.sql';
```

ou bien `clients.sql`, puis `articles.sql`, puis `achats.sql`.

**Justification :** chaque achat fait référence à un article (`aid`) et à un client (`cid`) par ses clés étrangères. À cause de la **contrainte d'intégrité référentielle**, le SGBD refuse d'enregistrer un achat dont l'article ou le client n'existe pas encore.

> ⚠️ **Liberté prise :** les fichiers `achats.sql`, `articles.sql` et `clients.sql` ne sont pas fournis. Pour illustrer la réponse, on tape à la place quelques `INSERT INTO` avec des données inventées (non demandé).

*Un achat avant que l'article et le client existent : refusé par H2.*

```sql
INSERT INTO Achats VALUES (10, 1, '2026-11-14', 2);
```

*Résultat dans la console H2 :*

```text
Erreur : Intégrité référentielle violation de contrainte: "CONSTRAINT_72: PUBLIC.ACHATS FOREIGN KEY(AID) REFERENCES PUBLIC.ARTICLES(AID) (10)"
Referential integrity constraint violation: "CONSTRAINT_72: PUBLIC.ACHATS FOREIGN KEY(AID) REFERENCES PUBLIC.ARTICLES(AID) (10)"; SQL statement:
INSERT INTO Achats VALUES (10, 1, '2026-11-14', 2) [23506-252]
```

*Dans le bon ordre : l'article et le client d'abord, puis les achats. Le même article acheté à une autre date est accepté (question 1).*

```sql
INSERT INTO Articles VALUES (10, 'Papeterie', 2.5);
INSERT INTO Clients VALUES (1, 'Durand', '1990-05-17');
INSERT INTO Achats VALUES (10, 1, '2026-11-14', 2);
INSERT INTO Achats VALUES (10, 1, '2026-11-20', 1);
SELECT * FROM Achats;
```

*Résultat dans la console H2 :*

```text
INSERT INTO Articles VALUES (10, 'Papeterie', 2.5);
(Update count: 1)

INSERT INTO Clients VALUES (1, 'Durand', '1990-05-17');
(Update count: 1)

INSERT INTO Achats VALUES (10, 1, '2026-11-14', 2);
(Update count: 1)

INSERT INTO Achats VALUES (10, 1, '2026-11-20', 1);
(Update count: 1)

SELECT * FROM Achats;
AID | CID | DATEACHAT  | QUANTITÉ
10  | 1   | 2026-11-14 | 2
10  | 1   | 2026-11-20 | 1
(2 rows)
```

*Même client, même article, même date : refusé (contrainte de clé primaire, question 1).*

```sql
INSERT INTO Achats VALUES (10, 1, '2026-11-14', 5);
```

*Résultat dans la console H2 :*

```text
Erreur : Violation d'index unique ou clé primaire: "PUBLIC.CONSTRAINT_7 INDEX PUBLIC.PRIMARY_KEY_7 ON PUBLIC.ACHATS(AID, CID, DATEACHAT) VALUES ( /* key:2 */ 10, 1, DATE '2026-11-14')"
Unique index or primary key violation: "PUBLIC.CONSTRAINT_7 INDEX PUBLIC.PRIMARY_KEY_7 ON PUBLIC.ACHATS(AID, CID, DATEACHAT) VALUES ( /* key:2 */ 10, 1, DATE '2026-11-14')"; SQL statement:
INSERT INTO Achats VALUES (10, 1, '2026-11-14', 5) [23505-252]
```

#### 💡 EXPLICATION PÉDAGOGIQUE

- **Contrainte d'intégrité référentielle (cours, page 6) :** une clé étrangère doit toujours avoir pour valeur une valeur **qui existe** dans l'attribut référencé. Un achat ne peut pas concerner un client « fantôme » : H2 répond « Intégrité référentielle violation de contrainte ».
- **Règle pratique :** on remplit d'abord les relations **référencées** (celles vers lesquelles pointent les clés étrangères), puis celles qui y font référence. C'est le même ordre que pour les `CREATE TABLE`.
- Pour **supprimer**, c'est l'inverse : on vide `Achats` avant `Articles` et `Clients`, sinon H2 refuse de supprimer un client qui a encore des achats.